# Лабораторна робота №3 — Візуалізація даних

**Датасет:** [Heart Disease — UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/45/heart+disease)

**Характеристики датасету:**
- Dataset Characteristics: Multivariate
- Attribute Characteristics: Categorical, Integer, Real
- Number of Attributes: 13 + target
- Has Missing Values: Yes


## 0. Імпорт бібліотек

In [3]:
import urllib.request
import zipfile
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

# Стиль графіків
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

print("Бібліотеки імпортовано")

Бібліотеки імпортовано


## 1. Завантаження датасету

Датасет Heart Disease містить медичні показники пацієнтів та ознаку наявності серцевого захворювання. Завантаження відбувається автоматично з UCI Repository.

In [5]:
DATASET_URL  = "https://archive.ics.uci.edu/static/public/45/heart+disease.zip"
DATASET_ZIP  = "heart_disease.zip"
DATA_DIR     = "heart_data"
TARGET_FILE  = os.path.join(DATA_DIR, "processed.cleveland.data")

def download_dataset():
    if os.path.exists(TARGET_FILE):
        print("Датасет вже завантажено.")
        return
    print("Завантаження датасету...")
    os.makedirs(DATA_DIR, exist_ok=True)
    req = urllib.request.Request(DATASET_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r:
        with open(DATASET_ZIP, "wb") as f:
            f.write(r.read())
    print("Розпакування...")
    with zipfile.ZipFile(DATASET_ZIP, "r") as z:
        z.extractall(DATA_DIR)
    os.remove(DATASET_ZIP)
    print("Готово.")

download_dataset()

Завантаження датасету...
Розпакування...
Готово.


## 2. Завантаження та Data Cleaning

**Атрибути датасету:**
| # | Назва | Тип | Опис |
|---|-------|-----|------|
| 1 | age | Integer | Вік пацієнта |
| 2 | sex | Categorical | Стать (1=чоловік, 0=жінка) |
| 3 | cp | Categorical | Тип болю у грудях (0–3) |
| 4 | trestbps | Integer | Артеріальний тиск у спокої (мм рт. ст.) |
| 5 | chol | Integer | Рівень холестерину (мг/дл) |
| 6 | fbs | Categorical | Цукор натще > 120 мг/дл (1=так) |
| 7 | restecg | Categorical | Результат ЕКГ у спокої (0–2) |
| 8 | thalach | Integer | Максимальна ЧСС |
| 9 | exang | Categorical | Стенокардія при навантаженні (1=так) |
| 10 | oldpeak | Real | Депресія ST при навантаженні |
| 11 | slope | Categorical | Нахил сегмента ST |
| 12 | ca | Integer | Кількість уражених судин (0–3) |
| 13 | thal | Categorical | Таласемія (3/6/7) |
| 14 | target | Categorical | Наявність захворювання (0=ні, 1–4=так) |


In [7]:
COLUMNS = [
    "age","sex","cp","trestbps","chol","fbs",
    "restecg","thalach","exang","oldpeak","slope","ca","thal","target"
]

df = pd.read_csv(TARGET_FILE, header=None, names=COLUMNS, na_values="?")

print(f"Розмір датасету: {df.shape}")
print(f"Пропуски:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
df.head()

Розмір датасету: (303, 14)
Пропуски:
ca      4
thal    2
dtype: int64


In [8]:
# Бінаризація target: 0 = здоровий, 1 = хворий
df["target"] = (df["target"] > 0).astype(int)

# Заповнення пропусків медіаною
for col in ["ca", "thal"]:
    df[col] = df[col].fillna(df[col].median())

# Мітки для читабельності
df["sex_label"]    = df["sex"].map({1: "Чоловік", 0: "Жінка"})
df["disease_label"]= df["target"].map({0: "Здоровий", 1: "Хворий"})
df["cp_label"]     = df["cp"].map({1:"Типова стенокардія", 2:"Атипова стенокардія",
                                    3:"Неангінальний біль", 4:"Безсимптомний"})

print(f"Після cleaning — пропусків: {df.isnull().sum().sum()}")
print(f"Розподіл за target: Здоровий={( df['target']==0).sum()}, Хворий={(df['target']==1).sum()}")
df.describe().round(2)

Після cleaning — пропусків: 0
Розподіл за target: Здоровий=164, Хворий=139


,age,trestbps,chol,thalach,oldpeak,ca
count,303.00,303.00,303.00,303.00,303.00,303.00
mean,54.44,131.69,246.69,149.61,1.04,0.67
std,9.04,17.60,51.78,22.88,1.16,0.94
min,29.00,94.00,126.00,71.00,0.00,0.00
max,77.00,200.00,564.00,202.00,6.20,3.00


## 3. Графіки

### Графік 1 — Scatter Plot: Вік vs Максимальна ЧСС

Залежність максимальної частоти серцевих скорочень від віку пацієнта з розбивкою за наявністю захворювання.

In [10]:
fig, ax = plt.subplots(figsize=(9, 6))

colors = {"Здоровий": "#2196F3", "Хворий": "#F44336"}
for label, group in df.groupby("disease_label"):
    ax.scatter(group["age"], group["thalach"],
               c=colors[label], label=label, alpha=0.7, edgecolors="white", s=70)

# Лінія тренду
for label, group in df.groupby("disease_label"):
    m, b, *_ = stats.linregress(group["age"], group["thalach"])
    x = np.linspace(group["age"].min(), group["age"].max(), 100)
    ax.plot(x, m*x + b, color=colors[label], linewidth=2, linestyle="--", alpha=0.8)

ax.set_title("Вік vs Максимальна ЧСС", fontsize=14, fontweight="bold")
ax.set_xlabel("Вік (роки)")
ax.set_ylabel("Максимальна ЧСС (уд/хв)")
ax.legend(title="Стан пацієнта")
plt.tight_layout()
plt.savefig("plot1_scatter_age_thalach.png", dpi=120)
plt.show()
print("З віком максимальна ЧСС знижується. У хворих пацієнтів ЧСС нижча при тому ж віці.")

З віком максимальна ЧСС знижується. У хворих пацієнтів ЧСС нижча при тому ж віці.


### Графік 2 — Гістограма: Рівень холестерину по 5 діапазонах

Розподіл пацієнтів за рівнем холестерину у 5 заданих діапазонах.

In [12]:
bins = [0, 200, 240, 280, 320, 600]
labels = ["<200
(Норма)", "200–240
(Межа)", "240–280
(Підвищений)", "280–320
(Високий)", ">320
(Критичний)"]

df["chol_range"] = pd.cut(df["chol"], bins=bins, labels=labels)
counts = df["chol_range"].value_counts().reindex(labels)

palette = ["#4CAF50","#8BC34A","#FFC107","#FF5722","#F44336"]
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(labels, counts.values, color=palette, edgecolor="white", linewidth=1.2, width=0.6)

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha="center", va="bottom", fontweight="bold", fontsize=12)

ax.set_title("Розподіл пацієнтів за рівнем холестерину (5 діапазонів)", fontsize=14, fontweight="bold")
ax.set_xlabel("Діапазон холестерину (мг/дл)")
ax.set_ylabel("Кількість пацієнтів")
ax.set_ylim(0, counts.max() + 15)
plt.tight_layout()
plt.savefig("plot2_histogram_chol.png", dpi=120)
plt.show()
print(f"Розподіл: {dict(zip(labels, counts.values))}")

Розподіл: {'<200\n(Норма)': 43, '200–240\n(Межа)': 91, '240–280\n(Підвищений)': 93, '280–320\n(Високий)': 48, '>320\n(Критичний)': 28}


### Графік 3 — Violin Plot: Розподіл віку за статтю та захворюванням

Багатовимірна візуалізація: розподіл віку з урахуванням статі та наявності захворювання (за прикладами зі статті про multi-dimensional data).

In [14]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.violinplot(data=df, x="sex_label", y="age", hue="disease_label",
               split=True, inner="quart", palette={"Здоровий":"#2196F3","Хворий":"#F44336"},
               ax=ax)

ax.set_title("Розподіл віку за статтю та наявністю захворювання", fontsize=14, fontweight="bold")
ax.set_xlabel("Стать")
ax.set_ylabel("Вік (роки)")
ax.legend(title="Стан пацієнта")
plt.tight_layout()
plt.savefig("plot3_violin_age_sex.png", dpi=120)
plt.show()
print("Violin plot показує повний розподіл: медіану, квартилі та форму розподілу.")

Violin plot показує повний розподіл: медіану, квартилі та форму розподілу.


### Графік 4 — Heatmap: Кореляційна матриця числових атрибутів

Візуалізація кореляцій між усіма числовими атрибутами датасету (multi-dimensional підхід).

In [16]:
num_cols = ["age","trestbps","chol","thalach","oldpeak","ca","target"]
corr = df[num_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))  # показуємо лише нижній трикутник
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax,
            cbar_kws={"shrink": 0.8})

ax.set_title("Кореляційна матриця числових атрибутів", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plot4_heatmap_corr.png", dpi=120)
plt.show()
print("Найсильніша від'ємна кореляція з target: thalach (-0.42), oldpeak (+0.43) — позитивна.")

Найсильніша від'ємна кореляція з target: thalach (-0.42), oldpeak (+0.43) — позитивна.


### Графік 5 — Pair Plot: Матриця парних залежностей

Pair plot для ключових числових атрибутів з розбивкою за наявністю захворювання — класичний інструмент multi-dimensional аналізу.

In [18]:
key_cols = ["age","thalach","chol","oldpeak","disease_label"]
g = sns.pairplot(df[key_cols], hue="disease_label",
                 palette={"Здоровий":"#2196F3","Хворий":"#F44336"},
                 diag_kind="kde", plot_kws={"alpha":0.6, "s":40})
g.fig.suptitle("Матриця парних залежностей ключових атрибутів", y=1.02,
               fontsize=14, fontweight="bold")
plt.savefig("plot5_pairplot.png", dpi=100, bbox_inches="tight")
plt.show()
print("Pair plot дозволяє побачити всі парні залежності одночасно.")

Pair plot дозволяє побачити всі парні залежності одночасно.


### Графік 6 — Box Plot: Артеріальний тиск за типом болю у грудях

Розподіл артеріального тиску в залежності від типу болю з урахуванням наявності захворювання.

In [20]:
cp_order = ["Типова стенокардія","Атипова стенокардія","Неангінальний біль","Безсимптомний"]
df_cp = df.dropna(subset=["cp_label"])

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df_cp, x="cp_label", y="trestbps", hue="disease_label",
            order=cp_order,
            palette={"Здоровий":"#2196F3","Хворий":"#F44336"},
            width=0.6, ax=ax)

ax.set_title("Артеріальний тиск за типом болю у грудях", fontsize=14, fontweight="bold")
ax.set_xlabel("Тип болю у грудях")
ax.set_ylabel("Артеріальний тиск у спокої (мм рт. ст.)")
ax.legend(title="Стан пацієнта")
plt.tight_layout()
plt.savefig("plot6_boxplot_bp_cp.png", dpi=120)
plt.show()
print("При безсимптомному типу болю хворі мають вищий середній тиск.")

При безсимптомному типу болю хворі мають вищий середній тиск.


### Графік 7 — Stacked Bar: Розподіл захворювання за віковими групами та статтю

Частка хворих та здорових у кожній віковій групі з розбивкою за статтю.

In [22]:
df["age_group"] = pd.cut(df["age"],
                          bins=[29,40,50,60,70,80],
                          labels=["30–40","41–50","51–60","61–70","71–80"])

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, (sex, group) in zip(axes, df.groupby("sex_label")):
    ct = group.groupby(["age_group","disease_label"]).size().unstack(fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.plot(kind="bar", stacked=True, ax=ax,
                color=["#2196F3","#F44336"], edgecolor="white", width=0.6)
    ax.set_title(f"{sex}", fontsize=13, fontweight="bold")
    ax.set_xlabel("Вікова група")
    ax.set_ylabel("Частка (%)")
    ax.set_xticklabels(ct_pct.index, rotation=0)
    ax.legend(title="Стан", labels=["Здоровий","Хворий"])
    ax.set_ylim(0, 110)

fig.suptitle("Частка захворювання за віковими групами та статтю", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plot7_stacked_age_sex.png", dpi=120)
plt.show()
print("У чоловіків після 50 років частка хворих суттєво зростає.")

У чоловіків після 50 років частка хворих суттєво зростає.


### Графік 8 — Line Plot: Середня ЧСС та холестерин за віком

Залежність середніх значень ключових показників від віку пацієнта.

In [24]:
age_stats = df.groupby("age")[["thalach","chol"]].mean().rolling(3, center=True).mean()

fig, ax1 = plt.subplots(figsize=(12, 6))

color1, color2 = "#2196F3", "#FF5722"
ax1.plot(age_stats.index, age_stats["thalach"], color=color1,
         linewidth=2.5, marker="o", markersize=4, label="Макс. ЧСС (лів. вісь)")
ax1.set_xlabel("Вік (роки)")
ax1.set_ylabel("Максимальна ЧСС (уд/хв)", color=color1)
ax1.tick_params(axis="y", labelcolor=color1)

ax2 = ax1.twinx()
ax2.plot(age_stats.index, age_stats["chol"], color=color2,
         linewidth=2.5, marker="s", markersize=4, linestyle="--", label="Холестерин (прав. вісь)")
ax2.set_ylabel("Холестерин (мг/дл)", color=color2)
ax2.tick_params(axis="y", labelcolor=color2)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc="upper right")

ax1.set_title("Середня ЧСС та рівень холестерину за віком (згладжено)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plot8_lineplot_age_trends.png", dpi=120)
plt.show()
print("З віком ЧСС знижується, а холестерин зростає до ~60 років, потім стабілізується.")

З віком ЧСС знижується, а холестерин зростає до ~60 років, потім стабілізується.


## Висновки

| # | Графік | Тип | Ключовий висновок |
|---|--------|-----|-------------------|
| 1 | Вік vs ЧСС | Scatter + trend | З віком ЧСС знижується; у хворих ЧСС нижча |
| 2 | Холестерин | Histogram (5 діапазонів) | Більшість пацієнтів у діапазоні 200–280 мг/дл |
| 3 | Вік за статтю | Violin Plot | Хворі чоловіки старші за здорових |
| 4 | Кореляції | Heatmap | thalach та oldpeak найсильніше пов'язані з target |
| 5 | Парні залежності | Pair Plot | oldpeak чітко розділяє хворих та здорових |
| 6 | Тиск за типом болю | Box Plot | При безсимптомному болю тиск вищий у хворих |
| 7 | Вік + стать | Stacked Bar | Після 50 у чоловіків різко зростає частка хворих |
| 8 | Тренди за віком | Line Plot | ЧСС падає, холестерин зростає до 60 років |
